# ECLIPSE · Notebook 01 — Data Exploration

Explore TESS sector 1 light curves. Visualize raw PDCSAP flux, denoised curves, and TLS period search output.

**Prerequisites:** `pip install -r requirements.txt` and a working MAST token in `.env`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0A1628'
matplotlib.rcParams['axes.facecolor'] = '#0A1628'
matplotlib.rcParams['text.color'] = '#E8EAF6'
matplotlib.rcParams['axes.labelcolor'] = '#E8EAF6'
matplotlib.rcParams['xtick.color'] = '#90A4AE'
matplotlib.rcParams['ytick.color'] = '#90A4AE'
matplotlib.rcParams['grid.color'] = '#1E3A5F'
matplotlib.rcParams['axes.edgecolor'] = '#1E3A5F'
print('ECLIPSE Data Exploration Notebook loaded.')

In [ ]:
from src.ingestion.tess_fetcher import TESSFetcher

# Fetch a known exoplanet host: WASP-18 TIC 261136679
fetcher = TESSFetcher(sector=1, output_dir='../data/raw')
TIC_ID = 261136679
raw = fetcher.get_lightcurve(TIC_ID)

if raw:
    print(f'Fetched TIC {TIC_ID}: {len(raw["time"])} cadences')
    print(f'Time range: {raw["time"].min():.2f} – {raw["time"].max():.2f} BTJD')
else:
    print('No data — check MAST credentials in .env')

In [ ]:
# Plot raw light curve
if raw:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(raw['time'], raw['flux'], '.', ms=1, color='#4FC3F7', alpha=0.4, label='Raw PDCSAP')
    ax.set_xlabel('BTJD (days)')
    ax.set_ylabel('Flux (e⁻/s)')
    ax.set_title(f'ECLIPSE — TIC {TIC_ID} Raw PDCSAP Light Curve (Sector 1)', color='#E8EAF6')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Apply full denoising pipeline
from src.preprocessing.denoising import full_denoising_pipeline, apply_quality_mask

if raw:
    time, flux, flux_err, cx, cy = apply_quality_mask(
        raw['time'], raw['flux'], raw.get('flux_err', np.ones_like(raw['flux'])*0.001),
        raw.get('quality', np.zeros(len(raw['time']), dtype=np.int16)),
        raw.get('centroid_x', np.zeros_like(raw['time'])),
        raw.get('centroid_y', np.zeros_like(raw['time']))
    )
    t_clean, f_clean, e_clean = full_denoising_pipeline(time, flux, flux_err)
    print(f'After denoising: {len(t_clean)} cadences (from {len(time)})')

    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    axes[0].plot(time, flux/np.nanmedian(flux), '.', ms=1, color='#4FC3F7', alpha=0.3)
    axes[0].set_title('Before denoising', color='#E8EAF6')
    axes[0].set_ylabel('Normalized Flux')

    axes[1].plot(t_clean, f_clean, '.', ms=1, color='#80DEEA', alpha=0.5)
    axes[1].set_title('After sigma-clip + Wotan biweight', color='#E8EAF6')
    axes[1].set_ylabel('Detrended Flux')
    axes[1].set_xlabel('BTJD (days)')

    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# TLS Period Search
from src.preprocessing.period_search import run_tls_search

if raw:
    tces = run_tls_search(t_clean, f_clean, tic_id=TIC_ID, sde_threshold=5.0)
    for i, tce in enumerate(tces):
        print(f'TCE {i+1}: P={tce.period:.4f}d, depth={tce.depth:.5f}, SDE={tce.snr:.2f}, N={tce.n_transits}')

In [ ]:
# Phase fold the best TCE
from src.preprocessing.phase_fold import phase_fold

if raw and tces:
    best = tces[0]
    gv, lv = phase_fold(t_clean, f_clean, best.period, best.t0, best.duration_days)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    phase_g = np.linspace(-0.5, 0.5, len(gv))
    axes[0].plot(phase_g, gv, '-', lw=0.8, color='#4FC3F7')
    axes[0].set_title(f'Global View — P={best.period:.4f}d', color='#E8EAF6')
    axes[0].set_xlabel('Phase'); axes[0].set_ylabel('Normalized Flux')

    phase_l = np.linspace(-1, 1, len(lv))
    axes[1].plot(phase_l, lv, '-', lw=1.2, color='#80DEEA')
    axes[1].set_title('Local View (transit detail)', color='#E8EAF6')
    axes[1].set_xlabel('Phase (transit durations)')

    for ax in axes: ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
# Explore multiple TIC IDs from sector catalog
from src.ingestion.catalog_loader import CatalogLoader

loader = CatalogLoader(labels_dir='../data/labels')
try:
    toi_df = loader.load_toi_catalog()
    print(f'TOI catalog: {len(toi_df)} entries')
    print(toi_df[['tic_id', 'eclipse_label', 'period']].head(10))
except Exception as e:
    print(f'TOI catalog not yet downloaded: {e}')
    print('Run: python -m src.ingestion.catalog_loader to download')